In [ ]:
# ---------------- Imports ----------------
import os
import json

import yaml
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import os

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,

    
})


In [ ]:
# ---------------- Args ----------------
METRICS = {
    "BAcc": "balanced_accuracy",
    "MSRP": "mean_supports_prob_on_refutes",
}
append_to_output_name = "bacc_msrp"


#METRICS = {
#    "BAcc": "balanced_accuracy",
#    "Mean Conf Correct": "mean_conf_correct",
#    "Mean Conf Incorrect": "mean_conf_incorrect",
#}
#append_to_output_name = "bacc_mean_conf"





MODEL_CHOICE = "meta-llama/Llama-3.1-8B-Instruct"
ALPHA_CHOICE = "0.3"
SAMPLES_CHOICE = "15k"
FILES = {
    "baseline": [
        "20260201t182154-20260128T2129-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
        "20260201t182416-20260130T1707-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
        "20260201t182637-20260130T1730-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1",
    ],

    "authoritative": [
        "20260201t181455-20260130T1258-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260201t181713-20260130T1617-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260201t181933-20260130T1641-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
    ],

    "consensus": [
        "20260201t182857-20260131T1055-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
        "20260201t183118-20260131T1118-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
        "20260201t183341-20260131T1141-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3",
    ],

    "prestige": [
        "20260201t183605-20260131T1206-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",
        "20260201t183824-20260131T1229-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",
        "20260201t184043-20260131T1251-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3",

    ],

    "emotional": [
        "20260201t184305-20260131T1314-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
        "20260201t184526-20260131T1336-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
        "20260201t184747-20260131T1359-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3",
    ],

    "sensationalist": [
        "20260201t185009-20260131T1421-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
        "20260201t185227-20260131T1444-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
        "20260201t185444-20260131T1506-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3",
    ],

    "random": [
        "20260506t202510-20260506T1840-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-random-0.3",
        "20260506t202658-20260506T1905-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-random-0.3",
        "20260506t202848-20260506T1929-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-random-0.3",

    ],
}




In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]


RESULTS_DIR = os.path.join(PROJ_STORE, "evaluation", "diagnostic-individual-tables", MODEL_CHOICE)

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "diagnostic-averages", MODEL_CHOICE)
os.makedirs(OUTPUT_DIR, exist_ok=True)





In [ ]:
# -------------------------
# Workspace
# -------------------------

def extract_model_name(filename):
    parts = filename.split("-")

    if len(parts) < 6:
        raise ValueError(f"Unexpected filename format: {filename}")

    # skip first two segments, take next 3
    return "-".join(parts[2:5])


# LOAD MSPR VALUES
def load_and_average(files):

    dfs = []
    model_names = set()

    for fname in files:
        path = os.path.join(RESULTS_DIR, f"{fname}.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(path)


        # extract + track model
        model_name = extract_model_name(fname)
        model_names.add(model_name)
        
        df = pd.read_csv(path)
        dfs.append(df)

    
    # enforce single model per group
    if len(model_names) != 1:
        raise ValueError(f"Inconsistent models in group: {model_names}")

    model_name = model_names.pop()
    
    full = pd.concat(dfs)

    # enforce categorical order
    #full["framing_type"] = pd.Categorical(
    #    full["framing_type"],
    #    categories=FRAMING_ORDER,
    #    ordered=True,
    #)

    # dynamically aggregate only selected metrics
    agg_dict = {metric_col: "mean" for metric_col in METRICS.values()}

    avg = (
        full
        .groupby(
            "framing_type",
            as_index=False,
            sort=True,
            observed=False,
        )
        .agg(agg_dict)
        .sort_values("framing_type")
    )

    return avg, model_name



In [ ]:
def build_summary_table(files_dict):

    model_tables = {}
    model_names_global = set()

    # ---- load each condition ----
    for condition_name, files in files_dict.items():

        df, model_name = load_and_average(files)

        model_tables[condition_name] = df.set_index("framing_type")
        model_names_global.add(model_name)

    # ---- ensure same model ----
    if len(model_names_global) != 1:
        raise ValueError(f"Different models across groups: {model_names_global}")

    # ---- collect framings ----
    all_framings = set().union(*[df.index for df in model_tables.values()])

    # ---- enforce required framings ----
    required = {"original", "OVERALL"}
    missing = required - all_framings
    if missing:
        raise ValueError(f"Missing required framings: {missing}")

    # ---- custom ordering ----
    middle = sorted(f for f in all_framings if f not in {"original", "OVERALL"})
    ordered_framings = ["original"] + middle + ["OVERALL"]

    # ---- build rows ----
    rows = []

    for framing in ordered_framings:

        row = {"Framing": framing}

        for condition_name, df in model_tables.items():

            for pretty_name, column_name in METRICS.items():

                value = df[column_name].get(framing, float("nan"))
                row[f"{condition_name}_{pretty_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# MAIN
table = build_summary_table(FILES)
print(table.to_string(index=False))






In [ ]:
# save

model_name = MODEL_CHOICE.split("/")[-1].lower()

out_csv = os.path.join(OUTPUT_DIR, f"{model_name}-{SAMPLES_CHOICE}-{ALPHA_CHOICE}-framing-matrix-{append_to_output_name}.csv".replace("_","-"))
table.to_csv(out_csv, index=False)

print("Saved:", out_csv)




In [ ]:
def extract_overall_tidy(matrix_df):

    # ---- ensure OVERALL exists ----
    if "OVERALL" not in matrix_df["Framing"].values:
        raise ValueError("OVERALL row not found in matrix")

    overall_row = matrix_df[matrix_df["Framing"] == "OVERALL"].iloc[0]

    rows = []

    for condition_name in FILES.keys():

        row = {"Framing": condition_name.capitalize()}

        for pretty_name in METRICS.keys():

            col = f"{condition_name}_{pretty_name}"

            if col not in overall_row:
                raise ValueError(f"Missing column: {col}")

            row[pretty_name] = overall_row[col]

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:

overall_only = extract_overall_tidy(table)

print(overall_only.to_string(index=False))



In [ ]:
# save

base, ext = os.path.splitext(out_csv)
out_csv_overall = f"{base}-overall{ext}"

overall_only.to_csv(out_csv_overall, index=False)

print("Saved:", out_csv_overall)


